# Data Analytics — Exercise 7.3: Pandas DataFrames & Merging
## Patrick Green-Holloway | Date: 2026-05-22

This notebook helps GlobalTrade Ltd. compare customer orders with shipment records.  
The goal is to see which orders shipped, which orders are still missing shipment info, and whether any shipment exists without a matching order.

## 1. Check the Folder and Load CSV Files

Before I start working with the data, I want to make sure the CSV files are in the same folder as this notebook.  
Then I will load `orders.csv` and `shipments.csv` into pandas DataFrames.

In [1]:
# I am importing the tools I need first.
# pandas is used to read CSV files and work with tables.
# os is just used here to check what files are in my folder.
import os
import pandas as pd

# This helps me check that my CSV files are in the right place.
print("Files in this folder:")
print(os.listdir("."))

# I am loading both CSV files into DataFrames.
df_orders = pd.read_csv("orders.csv")
df_shipments = pd.read_csv("shipments.csv")

Files in this folder:
['Week 7 Lab Workbook.pdf', 'Capstone 2 _Skill Check.pdf', 'Ex1B_NumPy_Random.ipynb', 'Ex1C_Operations_on_Arrays.ipynb', 'Class_Practice_Exercise_7_3.pdf', 'orders.csv', 'shipments.csv', 'DA_Exercise_7_3.ipynb', 'DA_Exercise_7_3_completed_student_comments.ipynb']


## 2. Inspect the Orders DataFrame

Now I am checking the orders table.  
This helps me see the first few rows, the table size, the data types, and whether anything is missing.

In [2]:
# I am previewing the first 5 rows so I can quickly see what the orders data looks like.
print("First 5 rows of df_orders:")
print(df_orders.head())

# Shape tells me the number of rows and columns.
print("\nShape of df_orders:")
print(df_orders.shape)

# dtypes tells me what kind of data each column has.
print("\nData types in df_orders:")
print(df_orders.dtypes)

# This checks if any values are missing in each column.
print("\nMissing values in df_orders:")
print(df_orders.isnull().sum())

First 5 rows of df_orders:
   order_id customer   product  quantity  unit_price
0       101     Chen    Laptop         2      899.99
1       102    Patel    Tablet         5      499.50
2       103   Müller   Monitor         1      349.00
3       104  Okonkwo  Keyboard        10       79.99
4       105   Santos   Headset         3      149.99

Shape of df_orders:
(10, 5)

Data types in df_orders:
order_id        int64
customer       object
product        object
quantity        int64
unit_price    float64
dtype: object

Missing values in df_orders:
order_id      0
customer      0
product       0
quantity      0
unit_price    0
dtype: int64


## 3. Inspect the Shipments DataFrame

Next I am checking the shipments table the same way.  
This is important because both tables need to be clean before I merge them.

In [3]:
# I am previewing the first 5 rows so I can quickly see what the shipment data looks like.
print("First 5 rows of df_shipments:")
print(df_shipments.head())

# Shape tells me the number of rows and columns.
print("\nShape of df_shipments:")
print(df_shipments.shape)

# dtypes tells me what kind of data each column has.
print("\nData types in df_shipments:")
print(df_shipments.dtypes)

# This checks if any values are missing in each column.
print("\nMissing values in df_shipments:")
print(df_shipments.isnull().sum())

First 5 rows of df_shipments:
   order_id   ship_date carrier      status
0       101  2024-01-10     DHL   Delivered
1       102  2024-01-12   FedEx   Delivered
2       103  2024-01-13     DHL  In Transit
3       104  2024-01-14     UPS   Delivered
4       105  2024-01-15   FedEx     Delayed

Shape of df_shipments:
(8, 4)

Data types in df_shipments:
order_id      int64
ship_date    object
carrier      object
status       object
dtype: object

Missing values in df_shipments:
order_id     0
ship_date    0
carrier      0
status       0
dtype: int64


## 4. Clean and Prepare the Data

For the orders table, I am creating a new `order_value` column.  
For the shipments table, I am changing `ship_date` into a real date format so pandas can understand it better.

In [4]:
# I am making a new column for the total value of each order.
# The formula is quantity times unit_price.
df_orders["order_value"] = df_orders["quantity"] * df_orders["unit_price"]

# This gives me quick number facts like average, min, and max.
print("Numeric summary for df_orders:")
print(df_orders.describe())

# I am converting ship_date from text into a real date.
df_shipments["ship_date"] = pd.to_datetime(df_shipments["ship_date"])

# I am checking the data types again to make sure ship_date changed correctly.
print("\nData types in df_shipments after date conversion:")
print(df_shipments.dtypes)

# This gives the total value of all orders together.
total_order_value = df_orders["order_value"].sum()
print(f"\nTotal order value: £{total_order_value:,.2f}")

Numeric summary for df_orders:
        order_id   quantity  unit_price  order_value
count   10.00000  10.000000   10.000000    10.000000
mean   105.50000   4.200000  245.743000   749.515000
std      3.02765   3.047768  273.023774   787.066644
min    101.00000   1.000000   29.990000    79.980000
25%    103.25000   2.000000   82.490000   267.190000
50%    105.50000   3.500000  134.990000   404.965000
75%    107.75000   5.750000  311.500000   779.910000
max    110.00000  10.000000  899.990000  2497.500000

Data types in df_shipments after date conversion:
order_id              int64
ship_date    datetime64[ns]
carrier              object
status               object
dtype: object

Total order value: £7,495.15


## 5. Inner Merge

An inner merge only keeps rows where the `order_id` exists in **both** tables.  
So this should only show orders that have matching shipment records.

In [5]:
# INNER merge means only matching order IDs from both tables stay.
df_inner = pd.merge(
    df_orders,
    df_shipments,
    on="order_id",
    how="inner"
)

print("INNER merge result:")
print(df_inner)

# My note: order IDs 101 through 107 appear because they exist in both files.
# Orders 108, 109, and 110 do not show because they do not have shipments.
# Shipment 111 does not show because it does not have an order record.
print(f"\nInner merge shape: {df_inner.shape}")

INNER merge result:
   order_id customer   product  ...  ship_date  carrier      status
0       101     Chen    Laptop  ... 2024-01-10      DHL   Delivered
1       102    Patel    Tablet  ... 2024-01-12    FedEx   Delivered
2       103   Müller   Monitor  ... 2024-01-13      DHL  In Transit
3       104  Okonkwo  Keyboard  ... 2024-01-14      UPS   Delivered
4       105   Santos   Headset  ... 2024-01-15    FedEx     Delayed
5       106  Larsson    Webcam  ... 2024-01-16      UPS   Delivered
6       107      Kim     Mouse  ... 2024-01-18      DHL  In Transit

[7 rows x 9 columns]

Inner merge shape: (7, 9)


## 6. Left Merge

A left merge keeps **all orders** from the orders table.  
If an order does not have a shipment, the shipment columns will show missing values.

In [6]:
# LEFT merge keeps every order, even if it has no shipment information.
df_left = pd.merge(
    df_orders,
    df_shipments,
    on="order_id",
    how="left"
)

print("LEFT merge result:")
print(df_left)

# I am finding orders where carrier is missing.
# These are the orders that have not shipped yet.
unshipped_orders = df_left[df_left["carrier"].isna()]

print("\nUnshipped orders:")
print(unshipped_orders[["order_id", "customer", "product", "order_value"]])

print(f"\nUnshipped order IDs: {list(unshipped_orders['order_id'])}")

LEFT merge result:
   order_id  customer          product  ...  ship_date  carrier      status
0       101      Chen           Laptop  ... 2024-01-10      DHL   Delivered
1       102     Patel           Tablet  ... 2024-01-12    FedEx   Delivered
2       103    Müller          Monitor  ... 2024-01-13      DHL  In Transit
3       104   Okonkwo         Keyboard  ... 2024-01-14      UPS   Delivered
4       105    Santos          Headset  ... 2024-01-15    FedEx     Delayed
5       106   Larsson           Webcam  ... 2024-01-16      UPS   Delivered
6       107       Kim            Mouse  ... 2024-01-18      DHL  In Transit
7       108   O'Brien  Docking Station  ...        NaT      NaN         NaN
8       109     Rossi              SSD  ...        NaT      NaN         NaN
9       110  Nakamura          USB Hub  ...        NaT      NaN         NaN

[10 rows x 9 columns]

Unshipped orders:
   order_id  customer          product  order_value
7       108   O'Brien  Docking Station       199.00

## 7. Right Merge

A right merge keeps **all shipment records** from the shipments table.  
This helps me find shipment records that do not match an order.

In [7]:
# RIGHT merge keeps every shipment, even if it does not have a matching order.
df_right = pd.merge(
    df_orders,
    df_shipments,
    on="order_id",
    how="right"
)

print("RIGHT merge result:")
print(df_right)

# I am finding shipments where the customer is missing.
# That means the shipment does not have a matching order record.
orphan_shipments = df_right[df_right["customer"].isna()]

print("\nOrphan shipment:")
print(orphan_shipments[["order_id", "ship_date", "carrier", "status"]])

RIGHT merge result:
   order_id customer   product  ...  ship_date  carrier      status
0       101     Chen    Laptop  ... 2024-01-10      DHL   Delivered
1       102    Patel    Tablet  ... 2024-01-12    FedEx   Delivered
2       103   Müller   Monitor  ... 2024-01-13      DHL  In Transit
3       104  Okonkwo  Keyboard  ... 2024-01-14      UPS   Delivered
4       105   Santos   Headset  ... 2024-01-15    FedEx     Delayed
5       106  Larsson    Webcam  ... 2024-01-16      UPS   Delivered
6       107      Kim     Mouse  ... 2024-01-18      DHL  In Transit
7       111      NaN       NaN  ... 2024-01-19    FedEx   Delivered

[8 rows x 9 columns]

Orphan shipment:
   order_id  ship_date carrier     status
7       111 2024-01-19   FedEx  Delivered


## 8. Outer Merge

An outer merge keeps **everything** from both tables.  
This is useful when I want a full audit and do not want to lose any rows.

In [8]:
# OUTER merge keeps all orders and all shipments.
df_outer = pd.merge(
    df_orders,
    df_shipments,
    on="order_id",
    how="outer"
)

print("OUTER merge result:")
print(df_outer)

# I am checking the shape and total missing values.
print(f"\nOuter merge shape: {df_outer.shape}")
print(f"Total NaN values in outer merge: {df_outer.isnull().sum().sum()}")

OUTER merge result:
    order_id  customer          product  ...  ship_date  carrier      status
0        101      Chen           Laptop  ... 2024-01-10      DHL   Delivered
1        102     Patel           Tablet  ... 2024-01-12    FedEx   Delivered
2        103    Müller          Monitor  ... 2024-01-13      DHL  In Transit
3        104   Okonkwo         Keyboard  ... 2024-01-14      UPS   Delivered
4        105    Santos          Headset  ... 2024-01-15    FedEx     Delayed
5        106   Larsson           Webcam  ... 2024-01-16      UPS   Delivered
6        107       Kim            Mouse  ... 2024-01-18      DHL  In Transit
7        108   O'Brien  Docking Station  ...        NaT      NaN         NaN
8        109     Rossi              SSD  ...        NaT      NaN         NaN
9        110  Nakamura          USB Hub  ...        NaT      NaN         NaN
10       111       NaN              NaN  ... 2024-01-19    FedEx   Delivered

[11 rows x 9 columns]

Outer merge shape: (11, 9)
Total

## 9. Add a Shipped Flag and Calculate Fulfilment Rate

Now I am using the left merge because it includes all customer orders.  
I will create a True/False column called `shipped` to show whether each order has shipment info.

In [9]:
# If carrier is not missing, then the order shipped.
# If carrier is missing, then the order did not ship yet.
df_left["shipped"] = df_left["carrier"].notna()

print("Orders with shipped flag:")
print(df_left[["order_id", "customer", "order_value", "shipped"]])

# This counts how many orders are shipped and not shipped.
print("\nShipped vs unshipped count:")
print(df_left["shipped"].value_counts())

# This calculates the fulfilment rate as a percentage.
shipped_count = df_left["shipped"].sum()
total_orders = len(df_left)
fulfilment_rate = shipped_count / total_orders * 100

print(f"\nFulfilment rate: {fulfilment_rate:.2f}%")

Orders with shipped flag:
   order_id  customer  order_value  shipped
0       101      Chen      1799.98     True
1       102     Patel      2497.50     True
2       103    Müller       349.00     True
3       104   Okonkwo       799.90     True
4       105    Santos       449.97     True
5       106   Larsson       359.96     True
6       107       Kim       239.92     True
7       108   O'Brien       199.00    False
8       109     Rossi       719.94    False
9       110  Nakamura        79.98    False

Shipped vs unshipped count:
shipped
True     7
False    3
Name: count, dtype: int64

Fulfilment rate: 70.00%


## 10. GroupBy Summary on Inner Merge

For this part, I am using the inner merge because it only contains orders that fully matched with shipments.  
I will summarize order value by delivery status and carrier.

In [10]:
# This shows total, average, and count of order value by shipment status.
status_summary = df_inner.groupby("status")["order_value"].agg(["sum", "mean", "count"])

print("Order value summary by status:")
print(status_summary.round(2))

# This shows total order value handled by each carrier.
carrier_totals = (
    df_inner.groupby("carrier")["order_value"]
    .sum()
    .sort_values(ascending=False)
)

print("\nCarrier totals:")
print(carrier_totals)

# Since the carrier totals are sorted highest to lowest, index[0] gives the top carrier.
top_carrier = carrier_totals.index[0]
top_carrier_value = carrier_totals.iloc[0]

print(f"\nTop carrier: {top_carrier} — £{top_carrier_value:,.2f}")

# This shows how many units were sent out for each status.
units_per_status = df_inner.groupby("status")["quantity"].sum()

print("\nUnits dispatched by status:")
print(units_per_status)

Order value summary by status:
                sum     mean  count
status                             
Delayed      449.97   449.97      1
Delivered   5457.34  1364.34      4
In Transit   588.92   294.46      2

Carrier totals:
carrier
FedEx    2947.47
DHL      2388.90
UPS      1159.86
Name: order_value, dtype: float64

Top carrier: FedEx — £2,947.47

Units dispatched by status:
status
Delayed        3
Delivered     21
In Transit     9
Name: quantity, dtype: int64


## 11. Summary

The fulfilment rate is **70.00%**, which means **7 out of 10 orders** have shipment records.  
Orders **108, 109, and 110** are still unshipped because they appear in the orders file but not the shipments file.  
The top carrier is **FedEx** with **£2,947.47** in matched order value, and shipment **111** should be checked because it has no matching order record.

# Extension Challenges

The main requirements are finished above.  
These extra sections add more practice with merge indicators, dates, exporting CSV files, and pivot tables.

## Extension 1: Reload Shipments with `parse_dates`

Here I am loading the shipments file again, but this time pandas converts the date column while reading the file.

In [11]:
# This is another way to make ship_date a real date.
# Instead of converting it after loading, pandas does it while reading the CSV.
df_shipments_dates = pd.read_csv("shipments.csv", parse_dates=["ship_date"])

print("Data types after using parse_dates:")
print(df_shipments_dates.dtypes)

Data types after using parse_dates:
order_id              int64
ship_date    datetime64[ns]
carrier              object
status               object
dtype: object


Using `parse_dates` is a shortcut.  
It saves me one extra line because I do not need to use `pd.to_datetime()` after loading the file.

## Extension 2: Merge Indicator

The merge indicator adds a `_merge` column.  
This helps me quickly see whether each row matched or only came from one side.

In [12]:
# indicator=True adds a column that explains where each row came from.
df_left_indicator = pd.merge(
    df_orders,
    df_shipments,
    on="order_id",
    how="left",
    indicator=True
)

print("Left merge with indicator:")
print(df_left_indicator[["order_id", "customer", "carrier", "_merge"]])

print("\nMerge indicator counts:")
print(df_left_indicator["_merge"].value_counts())

Left merge with indicator:
   order_id  customer carrier     _merge
0       101      Chen     DHL       both
1       102     Patel   FedEx       both
2       103    Müller     DHL       both
3       104   Okonkwo     UPS       both
4       105    Santos   FedEx       both
5       106   Larsson     UPS       both
6       107       Kim     DHL       both
7       108   O'Brien     NaN  left_only
8       109     Rossi     NaN  left_only
9       110  Nakamura     NaN  left_only

Merge indicator counts:
_merge
both          7
left_only     3
right_only    0
Name: count, dtype: int64


In the `_merge` column, `both` means the order matched with a shipment.  
`left_only` means the order came from the orders file only, so it is unshipped in this data.

## Extension 3: Ship Month Summary

Now I am pulling out the month name from the shipment date and grouping by month.

In [13]:
# I am creating a month name column from the shipment date.
df_inner["ship_month"] = df_inner["ship_date"].dt.month_name()

# This totals matched order value by shipment month.
month_summary = (
    df_inner.groupby("ship_month")["order_value"]
    .sum()
    .sort_values(ascending=False)
)

print("Order value by ship month:")
print(month_summary)

# The first row is the highest because I sorted it from biggest to smallest.
top_month = month_summary.index[0]
print(f"\nHighest dispatch value month: {top_month}")

Order value by ship month:
ship_month
January    6496.23
Name: order_value, dtype: float64

Highest dispatch value month: January


## Extension 4: Simulate Fixing the Orphan Shipment

Order ID 111 exists in shipments but not in orders.  
Here I am adding a fake matching order record so I can see it appear in the inner merge.

In [14]:
# I am making a realistic order row for order_id 111.
new_order = pd.DataFrame({
    "order_id": [111],
    "customer": ["Garcia"],
    "product": ["Printer"],
    "quantity": [1],
    "unit_price": [249.99]
})

# I am calculating order_value for the new row too.
new_order["order_value"] = new_order["quantity"] * new_order["unit_price"]

# I am adding the new order to the original orders DataFrame.
df_orders_fixed = pd.concat([df_orders, new_order], ignore_index=True)

# Now I re-run the inner merge to check if order 111 matches.
df_inner_fixed = pd.merge(
    df_orders_fixed,
    df_shipments,
    on="order_id",
    how="inner"
)

print("Checking if order_id 111 is now in the inner merge:")
print(df_inner_fixed[df_inner_fixed["order_id"] == 111])

Checking if order_id 111 is now in the inner merge:
   order_id customer  product  ...  ship_date  carrier     status
7       111   Garcia  Printer  ... 2024-01-19    FedEx  Delivered

[1 rows x 9 columns]


Now order ID **111** appears in the inner merge because I added a matching order record.  
This shows how fixing missing data can change the merge results.

## Extension 5: Export Fulfilment Report

Now I am saving the left merge as a new CSV file.  
This would be useful if I wanted to send the fulfilment report to someone else.

In [15]:
# I am exporting the left merge to a new CSV file.
df_left.to_csv("fulfilment_report.csv", index=False)

# I am reading it back in to make sure it saved correctly.
df_report_check = pd.read_csv("fulfilment_report.csv")

print("Fulfilment report shape:")
print(df_report_check.shape)

print("\nFulfilment report columns:")
print(df_report_check.columns.tolist())

Fulfilment report shape:
(10, 10)

Fulfilment report columns:
['order_id', 'customer', 'product', 'quantity', 'unit_price', 'order_value', 'ship_date', 'carrier', 'status', 'shipped']


## Extension 6: Pivot Table by Carrier and Status

This pivot table shows order value split by carrier and shipment status.

In [16]:
# This pivot table compares carriers against shipment statuses.
carrier_status_pivot = df_inner.pivot_table(
    values="order_value",
    index="carrier",
    columns="status",
    aggfunc="sum",
    fill_value=0
)

print("Carrier/status pivot table:")
print(carrier_status_pivot)

Carrier/status pivot table:
status   Delayed  Delivered  In Transit
carrier                                
DHL         0.00    1799.98      588.92
FedEx     449.97    2497.50        0.00
UPS         0.00    1159.86        0.00


The pivot table makes it easier to compare carriers side by side.  
FedEx has the highest matched order value overall, but it also has the delayed order in this dataset.